# اجرای کامل scVI با scRareBench (GitHub main) در Google Colab

این نوت‌بوک یک اجرای انتها‌به‌انتها انجام می‌دهد:

1. نسخهٔ جاری `scRareBench` را مستقیماً از GitHub نصب می‌کند.
2. پکیج، `scvi-tools` و backend استاندارد `scib-metrics` را نصب می‌کند.
3. دیتاست رسمی `GSE194122` را از مرجع اصلی دریافت می‌کند.
4. تغییر کنترل‌شده مقاله را اعمال می‌کند و دیتاست ۸۹٬۱۹۹ سلولی را می‌سازد.
5. preprocessing مخصوص scVI را روی raw counts انجام می‌دهد.
6. scVI را آموزش می‌دهد و latent space سی‌بعدی تولید می‌کند.
7. latent و barcodeها را با ترتیب canonical دیتاست کنترل می‌کند.
8. همه خروجی‌های جاری scIB-compatible، معیارهای مقاله و معیارهای rare-cell اختصاصی را محاسبه می‌کند.
9. یک dashboard HTML کامل و standalone با تب‌های Overview، Metrics، scIB، Rare-cell، UMAP، Sankey، Reproducibility و Figures می‌سازد؛ Rare-cell Explorer عملکرد و failure را برای هر شش سناریوی rare و cell typeهای عضو جداگانه نشان می‌دهد.
10. گزارش PDF و ZIP کامل خروجی را می‌سازد؛ latent نیز با فلگ قابل اضافه/حذف است.

> پیش از اجرا در Colab از مسیر **Runtime → Change runtime type → T4 GPU** یا GPU قوی‌تر استفاده کنید.

نام پکیج همچنان `scRareBench` است. خروجی استاندارد با backend نسخه‌پین‌شده `scib-metrics` تولید می‌شود و از خروجی rare-cell جدا نگه داشته می‌شود.


## ۱. نصب مستقیم از GitHub

این notebook دیگر به ZIP محلی نیاز ندارد. ابتدا خود `scrarebench` بدون dependency نصب می‌شود؛ سپس `scrarebench.runtime.setup_notebook()` dependencyهای benchmark و روش انتخاب‌شده را با constraints، `pip check` و smoke-test مدیریت می‌کند.


In [ ]:
SCRAREBENCH_GITHUB = "git+https://github.com/amirhossein-alishahi/scRareBench_.git@main"
METHOD = "scvi"

# To pin a release/commit, replace @main above with @vX.Y.Z or @<commit-sha>.


In [ ]:
from __future__ import annotations

import subprocess
import sys
from pathlib import Path

# Bootstrap only the lightweight package code from GitHub. Dependencies are
# deliberately handled by scrarebench.runtime in the next step.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--no-deps", SCRAREBENCH_GITHUB
])

from scrarebench.runtime import print_install_report, setup_notebook

install_report = setup_notebook(METHOD, quiet=False)
print_install_report(install_report)

# Import only after dependency validation and the fresh-process smoke test pass.
import scrarebench as _scrarebench_install_check
import scib_metrics as _scib_metrics_install_check
print("scrarebench import path:", Path(_scrarebench_install_check.__file__).resolve())
print("scrarebench version:", _scrarebench_install_check.__version__)
print("scib-metrics version:", getattr(_scib_metrics_install_check, "__version__", "installed"))
import scvi as _method_install_check
print("scvi-tools version:", _method_install_check.__version__)


## ۲. imports، نسخه‌ها و GPU


In [ ]:
from __future__ import annotations

from pathlib import Path
import gc
import hashlib
import json
import os
import platform
import sys
import importlib
import shutil
import time
import warnings

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import torch
import yaml
import scvi


import scrarebench
from IPython.display import HTML, display

print("Python:", platform.python_version())
print("pandas:", pd.__version__)
print("scRareBench:", scrarebench.__version__)
print("scIB runtime compatibility: automatic (pandas 2/3 safe)")
print("scanpy:", sc.__version__)
print("anndata:", ad.__version__)
print("scvi-tools:", scvi.__version__)
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    warnings.warn("GPU فعال نیست. آموزش کامل scVI روی CPU بسیار کند خواهد بود.")


## ۴. تنظیمات اجرای رسمی scVI و benchmark

In [ ]:
# مسیرهای اجرا
WORK_DIR = Path("/content/scrarebench_scvi_run")
DATA_DIR = WORK_DIR / "data"
CACHE_DIR = DATA_DIR / "cache"
MODEL_DIR = WORK_DIR / "scvi_model"
RESULTS_DIR = WORK_DIR / "results" / "scVI"
ARTIFACT_DIR = WORK_DIR / "deliverable"

for directory in (WORK_DIR, DATA_DIR, CACHE_DIR, RESULTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

BATCH_KEY = "BATCH"
LABEL_KEY = "celltype"
COUNTS_LAYER = "counts"
LATENT_KEY = "X_scVI"
METHOD_NAME = "scVI"

# تنظیمات scVI
SEED = 42
N_HVG = 4000
N_LATENT = 30
N_HIDDEN = 128
N_LAYERS = 2
DROPOUT_RATE = 0.10
DISPERSION = "gene-batch"
GENE_LIKELIHOOD = "nb"
MAX_EPOCHS = 200
TRAIN_SIZE = 0.90
BATCH_SIZE = 256
EARLY_STOPPING = True
EARLY_STOPPING_PATIENCE = 20

# clustering رسمی rare/paper layer
N_NEIGHBORS = 15
REFERENCE_RESOLUTION = 1.0
RESOLUTION_SWEEP = (1.0,)
DISTANCE_METRIC = "euclidean"

# standard scIB-compatible layer
RUN_SCIB = True
SCIB_N_HVG = 4000
SCIB_REFERENCE_N_PCS = 50
SCIB_N_JOBS = 1
SCIB_PROGRESS_BAR = True
SCIB_INCLUDE_SILHOUETTE_BATCH = True
SCIB_REQUIRE_SUCCESS = True

REUSE_MATCHING_SCVI_MODEL = True
FORCE_REBUILD_DATASET = False
GENERATE_UMAP = True
DOWNLOAD_RESULT_ZIP = True
DOWNLOAD_STANDALONE_REPORT = False
GENERATE_INTERACTIVE_REPORT = True
GENERATE_PDF_REPORT = True
INCLUDE_LATENT_IN_BUNDLE = True

# بخش‌های dashboard تعاملی — همه به صورت پیش‌فرض فعال هستند.
# False کردن هر بخش، payload همان بخش را نیز از HTML حذف می‌کند و فایل سبک‌تر می‌شود.
HTML_INCLUDE_OVERVIEW = True
HTML_INCLUDE_METRICS = True
HTML_INCLUDE_SCIB = True
HTML_INCLUDE_RARE = True
HTML_INCLUDE_RARE_UMAP = True
HTML_INCLUDE_RARE_HEATMAPS = True
HTML_INCLUDE_RARE_SCENARIO_ANALYSIS = True
HTML_INCLUDE_UMAP = True
HTML_INCLUDE_SANKEY = True
HTML_INCLUDE_REPRODUCIBILITY = True
HTML_INCLUDE_STATIC_FIGURES = True
HTML_INCLUDE_CELL_IDS = True  # False => barcodeها از hover حذف می‌شوند و HTML سبک‌تر می‌شود.

print("Work directory:", WORK_DIR)


## ۵. دانلود و ساخت دیتاست رسمی مقاله

In [ ]:
from scrarebench import load_dataset

benchmark_h5ad = DATA_DIR / "gse194122_paper_main.h5ad"

# Dataset selector 0 == "gse194122": download original source when needed,
# apply only the benchmark-specific cell subsetting + six-scenario annotation,
# and return the 89,199-cell AnnData. No scVI preprocessing happens here.
adata = load_dataset(
    0,
    DATA_DIR,
    force_download=False,
    force_rebuild=FORCE_REBUILD_DATASET,
    strict_expected_counts=True,
)

manifest_path = benchmark_h5ad.with_suffix(".manifest.json")
dataset_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

assert adata.n_obs == 89_199, adata.n_obs
assert BATCH_KEY in adata.obs
assert LABEL_KEY in adata.obs
assert adata.obs_names.is_unique

print(adata)
print("Cells:", adata.n_obs)
print("Genes/features:", adata.n_vars)
print("Batches:", adata.obs[BATCH_KEY].nunique())
print("Cell types:", adata.obs[LABEL_KEY].nunique())
print("Cell-order hash:", dataset_manifest["cell_order_sha256"])

distribution_path = benchmark_h5ad.with_suffix(".distribution.csv")
display(pd.read_csv(distribution_path))


## ۶. آماده‌سازی ورودی scVI

این مرحله فقط متعلق به روش scVI است و فایل benchmark روی دیسک را تغییر نمی‌دهد:

- فقط ویژگی‌های RNA/GEX نگه داشته می‌شوند؛
- raw-count layer اعتبارسنجی می‌شود؛
- ۴۰۰۰ HVG با `seurat_v3` و با لحاظ batch انتخاب می‌شوند؛
- cell typeها در انتخاب ویژگی یا آموزش مدل استفاده نمی‌شوند.


In [ ]:
def sampled_count_diagnostics(matrix, max_values: int = 100_000, seed: int = 42):
    values = matrix.data if sp.issparse(matrix) else np.asarray(matrix).ravel()
    values = np.asarray(values)
    if len(values) > max_values:
        rng = np.random.default_rng(seed)
        values = values[rng.choice(len(values), size=max_values, replace=False)]
    values = values.astype(float, copy=False)
    finite = values[np.isfinite(values)]
    diagnostics = {
        "sampled_values": len(values),
        "finite_fraction": float(np.mean(np.isfinite(values))) if len(values) else 1.0,
        "minimum": float(np.min(finite)) if len(finite) else 0.0,
        "maximum": float(np.max(finite)) if len(finite) else 0.0,
        "nonnegative": bool(len(finite) == 0 or finite.min() >= 0),
        "integer_like": bool(len(finite) == 0 or np.allclose(finite, np.rint(finite), atol=1e-6)),
    }
    return diagnostics

# RNA/GEX feature selection
if "feature_types" in adata.var.columns:
    feature_types = adata.var["feature_types"].astype(str).str.upper()
    rna_mask = feature_types.eq("GEX").to_numpy()
    if rna_mask.sum() == 0:
        raise ValueError("No GEX features were found in adata.var['feature_types'].")
    adata_rna = adata[:, rna_mask].copy() if rna_mask.sum() != adata.n_vars else adata
else:
    adata_rna = adata

adata_rna.var_names_make_unique()

if COUNTS_LAYER not in adata_rna.layers:
    raise KeyError(
        f"Raw-count layer {COUNTS_LAYER!r} is missing. Available layers: {list(adata_rna.layers)}"
    )

count_diagnostics = sampled_count_diagnostics(adata_rna.layers[COUNTS_LAYER], seed=SEED)
print("Count diagnostics:", count_diagnostics)
if not count_diagnostics["nonnegative"] or not count_diagnostics["integer_like"]:
    raise ValueError("The selected matrix does not look like nonnegative integer raw counts.")

# Batch-aware HVG selection on raw counts
sc.pp.highly_variable_genes(
    adata_rna,
    layer=COUNTS_LAYER,
    flavor="seurat_v3",
    n_top_genes=min(N_HVG, adata_rna.n_vars),
    batch_key=BATCH_KEY,
    subset=False,
)

hvg_mask = adata_rna.var["highly_variable"].fillna(False).to_numpy()
hvg_names = adata_rna.var_names[hvg_mask]
if len(hvg_names) == 0:
    raise RuntimeError("No HVGs were selected.")

counts_hvg = adata_rna.layers[COUNTS_LAYER][:, hvg_mask]
if sp.issparse(counts_hvg):
    counts_hvg = counts_hvg.tocsr().astype(np.float32)
else:
    counts_hvg = np.asarray(counts_hvg, dtype=np.float32)

adata_scvi = ad.AnnData(
    X=counts_hvg,
    obs=adata_rna.obs.copy(),
    var=adata_rna.var.loc[hvg_names].copy(),
)
adata_scvi.var_names_make_unique()

assert np.array_equal(adata_scvi.obs_names.astype(str), adata.obs_names.astype(str))
assert adata_scvi.n_vars == min(N_HVG, adata_rna.n_vars)

print(adata_scvi)
print("Selected HVGs:", adata_scvi.n_vars)


## ۷. آموزش یا بازیابی مدل scVI

In [ ]:
def hash_strings(values) -> str:
    digest = hashlib.sha256()
    for value in values:
        digest.update(str(value).encode("utf-8"))
        digest.update(b"\0")
    return digest.hexdigest()

scvi_config = {
    "seed": SEED,
    "n_hvg": N_HVG,
    "n_latent": N_LATENT,
    "n_hidden": N_HIDDEN,
    "n_layers": N_LAYERS,
    "dropout_rate": DROPOUT_RATE,
    "dispersion": DISPERSION,
    "gene_likelihood": GENE_LIKELIHOOD,
    "max_epochs": MAX_EPOCHS,
    "train_size": TRAIN_SIZE,
    "batch_size": BATCH_SIZE,
    "early_stopping": EARLY_STOPPING,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "batch_key": BATCH_KEY,
    "counts_source": "adata_scvi.X built from benchmark counts layer",
}

model_manifest = {
    "cell_hash": hash_strings(adata_scvi.obs_names),
    "gene_hash": hash_strings(adata_scvi.var_names),
    "batch_hash": hash_strings(adata_scvi.obs[BATCH_KEY].astype(str)),
    "n_obs": adata_scvi.n_obs,
    "n_vars": adata_scvi.n_vars,
    "config": scvi_config,
    "scvi_version": scvi.__version__,
}
manifest_path = MODEL_DIR / "run_manifest.json"

scvi.settings.seed = SEED
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# X contains raw counts in adata_scvi, so no layer argument is needed.
scvi.model.SCVI.setup_anndata(adata_scvi, batch_key=BATCH_KEY)

can_reuse = False
if REUSE_MATCHING_SCVI_MODEL and MODEL_DIR.exists() and manifest_path.exists():
    try:
        can_reuse = json.loads(manifest_path.read_text(encoding="utf-8")) == model_manifest
    except Exception:
        can_reuse = False

if can_reuse:
    print("Loading matching cached scVI model:", MODEL_DIR)
    model = scvi.model.SCVI.load(
        MODEL_DIR,
        adata=adata_scvi,
        accelerator="auto",
        device="auto",
    )
    training_seconds = 0.0
else:
    if MODEL_DIR.exists():
        shutil.rmtree(MODEL_DIR)
    print(f"Training scVI on {adata_scvi.n_obs:,} cells and {adata_scvi.n_vars:,} HVGs")
    model = scvi.model.SCVI(
        adata_scvi,
        n_hidden=N_HIDDEN,
        n_latent=N_LATENT,
        n_layers=N_LAYERS,
        dropout_rate=DROPOUT_RATE,
        dispersion=DISPERSION,
        gene_likelihood=GENE_LIKELIHOOD,
        latent_distribution="normal",
    )
    start = time.perf_counter()
    model.train(
        max_epochs=MAX_EPOCHS,
        accelerator="auto",
        devices="auto",
        train_size=TRAIN_SIZE,
        validation_size=1.0 - TRAIN_SIZE,
        batch_size=BATCH_SIZE,
        early_stopping=EARLY_STOPPING,
        early_stopping_patience=EARLY_STOPPING_PATIENCE,
        check_val_every_n_epoch=1,
        enable_progress_bar=True,
    )
    training_seconds = time.perf_counter() - start
    model.save(MODEL_DIR, overwrite=True, save_anndata=False)
    manifest_path.write_text(json.dumps(model_manifest, indent=2), encoding="utf-8")

print(model)
print(f"Training time: {training_seconds / 60:.2f} minutes")


## ۸. استخراج latent، ثبت barcodeها و کنترل ترتیب

In [ ]:
from scrarebench.latent import attach_latent

X_scVI = np.asarray(
    model.get_latent_representation(adata=adata_scvi, give_mean=True),
    dtype=np.float32,
)
barcodes = adata_scvi.obs_names.astype(str).to_numpy()

if X_scVI.shape != (adata.n_obs, N_LATENT):
    raise RuntimeError(f"Unexpected latent shape: {X_scVI.shape}")
if not np.isfinite(X_scVI).all():
    raise ValueError("X_scVI contains NaN or infinite values.")

latent_path = WORK_DIR / "scVI_latent.npy"
barcodes_path = WORK_DIR / "scVI_cell_barcodes.npy"
hvg_path = WORK_DIR / "scVI_hvgs.txt"
config_path = WORK_DIR / "scVI_config.json"
history_path = WORK_DIR / "scVI_training_history.csv"

np.save(latent_path, X_scVI)
np.save(barcodes_path, barcodes)
hvg_path.write_text("\n".join(map(str, adata_scvi.var_names)), encoding="utf-8")
config_payload = {**scvi_config, "training_seconds": training_seconds}
config_path.write_text(json.dumps(config_payload, indent=2), encoding="utf-8")

history_frames = []
for key, values in getattr(model, "history", {}).items():
    frame = pd.DataFrame(values).copy()
    frame["metric"] = key
    frame["epoch"] = frame.index
    history_frames.append(frame.reset_index(drop=True))
history_df = pd.concat(history_frames, ignore_index=True) if history_frames else pd.DataFrame()
history_df.to_csv(history_path, index=False)

alignment_report = attach_latent(
    adata,
    X_scVI,
    key=LATENT_KEY,
    latent_barcodes=barcodes,
    allow_reorder=False,
    overwrite=True,
)

print("Latent:", X_scVI.shape, X_scVI.dtype)
print("Alignment report:", alignment_report)
print("Saved latent:", latent_path)
print("Saved barcodes:", barcodes_path)

# حافظه GPU و AnnData مخصوص آموزش دیگر برای benchmark لازم نیست.
del adata_scvi, model, counts_hvg
if adata_rna is not adata:
    del adata_rna
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## ۹. اجرای benchmark استاندارد scRareBench

In [ ]:
from scrarebench.evaluation import EvaluationConfig, evaluate_latent
from scrarebench.scib_backend import ScibEvaluationConfig

if RESULTS_DIR.exists():
    shutil.rmtree(RESULTS_DIR)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

benchmark_config = EvaluationConfig(
    method_name=METHOD_NAME,
    representation_key=LATENT_KEY,
    label_key=LABEL_KEY,
    batch_key=BATCH_KEY,
    reference_resolution=REFERENCE_RESOLUTION,
    resolution_sweep=RESOLUTION_SWEEP,
    n_neighbors=N_NEIGHBORS,
    distance_metric=DISTANCE_METRIC,
    random_state=SEED,
    overwrite=True,
    scib=ScibEvaluationConfig(
        enabled=RUN_SCIB,
        count_layer=COUNTS_LAYER,
        n_hvg=SCIB_N_HVG,
        reference_n_pcs=SCIB_REFERENCE_N_PCS,
        n_jobs=SCIB_N_JOBS,
        progress_bar=SCIB_PROGRESS_BAR,
        include_silhouette_batch=SCIB_INCLUDE_SILHOUETTE_BATCH,
        require_backend=SCIB_REQUIRE_SUCCESS,
    ),
)

result = evaluate_latent(adata, benchmark_config, RESULTS_DIR)

print("Benchmark completed.")
if result.scib is not None:
    runtime_versions = result.scib.reference_config.get("runtime_versions", {})
    compatibility_adjustments = result.scib.reference_config.get("runtime_compatibility_adjustments", [])
    print("scIB runtime versions:", runtime_versions)
    print("scIB compatibility adjustments:", compatibility_adjustments or "none required")
print("Reference cluster key:", result.cluster_keys[REFERENCE_RESOLUTION])
print("\nPaper-style overall/subset metrics")
display(result.subset_metrics.round(5))
print("\nRare-cell summary")
display(result.rare_summary.round(5))
print("\nRare-cell per-type metrics")
display(result.rare_metrics.round(5))

if result.scib is not None:
    print("\nStandard scIB-compatible aggregate scores")
    display(result.scib.aggregate_scores.round(5))
    print("\nStandard scIB-compatible individual metrics")
    display(result.scib.metrics_long.round(5))
    print("\nMetric applicability/status")
    display(result.scib.metric_status)


## ۱۰. افزودن UMAP و نمودار آموزش به گزارش self-contained

گزارش اصلی پکیج از قبل شامل تمام جداول scIB-compatible و rare-cell است. این مرحله فقط نمودارهای مختص اجرای scVI را اضافه و همان گزارش را بازسازی می‌کند.


In [ ]:
from scrarebench.reporting import write_html_report, write_interactive_report, write_pdf_report

figures_dir = RESULTS_DIR / "rare_cell" / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)
extra_figures = []

run_config = yaml.safe_load((RESULTS_DIR / "reproducibility" / "run_config.yaml").read_text(encoding="utf-8"))
neighbors_key = run_config["neighbors_key"]
reference_cluster_key = run_config["reference_cluster_key"]

if GENERATE_UMAP:
    sc.tl.umap(adata, neighbors_key=neighbors_key, random_state=SEED)
    adata.obsm["X_umap_scVI"] = adata.obsm["X_umap"].copy()

    for color_key, title, filename, size in [
        (LABEL_KEY, "scVI latent UMAP — reference cell types", "umap_scvi_cell_types.png", (13, 9)),
        (BATCH_KEY, "scVI latent UMAP — batches", "umap_scvi_batches.png", (11, 8)),
    ]:
        sc.pl.embedding(
            adata,
            basis="X_umap_scVI",
            color=color_key,
            title=title,
            frameon=False,
            show=False,
            legend_loc="right margin",
        )
        path = figures_dir / filename
        plt.gcf().set_size_inches(*size)
        plt.savefig(path, dpi=180, bbox_inches="tight")
        plt.close()
        extra_figures.append(path)

    rare_mask = adata.obs["scrarebench_is_rare"].astype(bool).to_numpy()
    rare_view = adata[rare_mask].copy()
    sc.pl.embedding(
        rare_view,
        basis="X_umap_scVI",
        color="scrarebench_scenario",
        title="Curated rare populations — six scenarios",
        frameon=False,
        show=False,
        legend_loc="right margin",
        size=24,
    )
    path = figures_dir / "umap_scvi_rare_scenarios.png"
    plt.gcf().set_size_inches(11, 8)
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.close()
    extra_figures.append(path)

if not history_df.empty:
    fig, ax = plt.subplots(figsize=(9, 5.5))
    plotted = 0
    for metric, frame in history_df.groupby("metric"):
        numeric_columns = [
            c for c in frame.columns
            if c not in {"metric", "epoch"} and pd.api.types.is_numeric_dtype(frame[c])
        ]
        if not numeric_columns:
            continue
        ax.plot(frame["epoch"], frame[numeric_columns[0]].to_numpy(dtype=float), label=str(metric), alpha=0.85)
        plotted += 1
    if plotted:
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Recorded value")
        ax.set_title("scVI training history")
        ax.grid(alpha=0.25)
        ax.legend(fontsize=7)
        fig.tight_layout()
        path = figures_dir / "scvi_training_history.png"
        fig.savefig(path, dpi=180, bbox_inches="tight")
        extra_figures.append(path)
    plt.close(fig)

base_figures = []
if result.scib is not None:
    base_figures.append(result.scib.files["metric_plot"])
base_figures.extend([
    result.files["rare_metric_heatmap"],
    result.files["rare_precision_recall"],
    result.files["failure_counts"],
])

report_metadata = {
    "method": METHOD_NAME,
    "representation_key": LATENT_KEY,
    "n_cells": adata.n_obs,
    "n_dimensions": adata.obsm[LATENT_KEY].shape[1],
    "label_key": LABEL_KEY,
    "batch_key": BATCH_KEY,
    "reference_resolution": REFERENCE_RESOLUTION,
    "n_neighbors": N_NEIGHBORS,
    "distance_metric": DISTANCE_METRIC,
    "benchmark_seed": SEED,
    "scVI_n_hvg": N_HVG,
    "scVI_n_latent": N_LATENT,
    "scVI_max_epochs": MAX_EPOCHS,
    "scVI_training_seconds": round(training_seconds, 2),
    "reference_cluster_key": reference_cluster_key,
    "cell_order_exact_match": alignment_report["exact_order_match"],
    "scib_backend": result.scib.backend if result.scib else "disabled",
    "scib_backend_version": result.scib.backend_version if result.scib else "n/a",
}

report_path = RESULTS_DIR / "report.html"
write_html_report(
    report_path,
    title="scRareBench report — scVI on GSE194122 paper benchmark",
    metadata=report_metadata,
    global_table=result.subset_metrics,
    rare_table=result.rare_metrics,
    figure_names=base_figures + extra_figures,
    scib_metrics=result.scib.metrics_long if result.scib else None,
    scib_aggregates=result.scib.aggregate_scores if result.scib else None,
    scib_status=result.scib.metric_status if result.scib else None,
    rare_summary=result.rare_summary,
    scenario_table=result.scenario_metrics,
)

print("Self-contained report:", report_path)
print("Report size (MB):", round(report_path.stat().st_size / 1024**2, 2))

interactive_report_path = RESULTS_DIR / "interactive_report.html"
pdf_report_path = RESULTS_DIR / "summary_report.pdf"

# تصاویر اضافی ساخته‌شده در notebook را هم به dashboard معرفی می‌کنیم.
for index, figure_path in enumerate(extra_figures):
    result.files[f"notebook_figure_{index:02d}"] = figure_path

HTML_REPORT_OPTIONS = {
    "include_overview": HTML_INCLUDE_OVERVIEW,
    "include_metrics": HTML_INCLUDE_METRICS,
    "include_scib": HTML_INCLUDE_SCIB,
    "include_rare": HTML_INCLUDE_RARE,
    "include_rare_umap": HTML_INCLUDE_RARE_UMAP,
    "include_rare_heatmaps": HTML_INCLUDE_RARE_HEATMAPS,
    "include_rare_scenario_analysis": HTML_INCLUDE_RARE_SCENARIO_ANALYSIS,
    "include_umap": HTML_INCLUDE_UMAP,
    "include_sankey": HTML_INCLUDE_SANKEY,
    "include_reproducibility": HTML_INCLUDE_REPRODUCIBILITY,
    "include_static_figures": HTML_INCLUDE_STATIC_FIGURES,
    "include_cell_ids": HTML_INCLUDE_CELL_IDS,
}

if GENERATE_INTERACTIVE_REPORT:
    write_interactive_report(
        adata,
        result,
        interactive_report_path,
        representation_key=LATENT_KEY,
        label_key=LABEL_KEY,
        batch_key=BATCH_KEY,
        umap_key="X_umap_scVI" if "X_umap_scVI" in adata.obsm else None,
        **HTML_REPORT_OPTIONS,
    )

if GENERATE_PDF_REPORT:
    write_pdf_report(
        adata,
        result,
        pdf_report_path,
        representation_key=LATENT_KEY,
    )

print("Static HTML report:", report_path)
print("Interactive HTML report:", interactive_report_path if GENERATE_INTERACTIVE_REPORT else "disabled")
print("PDF report:", pdf_report_path if GENERATE_PDF_REPORT else "disabled")


## ۱۱. نمایش گزارش و بررسی فایل‌های خروجی در Colab

In [ ]:
display(HTML(report_path.read_text(encoding="utf-8")))

print("\nGenerated output files:")
for path in sorted(RESULTS_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(RESULTS_DIR))


## ۱۲. بسته‌بندی و دانلود خروجی نهایی

In [ ]:
from scrarebench.reporting import create_report_bundle

if ARTIFACT_DIR.exists():
    shutil.rmtree(ARTIFACT_DIR)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

if (RESULTS_DIR / "interactive_report.html").exists():
    shutil.copy2(RESULTS_DIR / "interactive_report.html", ARTIFACT_DIR / "interactive_report.html")
if (RESULTS_DIR / "summary_report.pdf").exists():
    shutil.copy2(RESULTS_DIR / "summary_report.pdf", ARTIFACT_DIR / "summary_report.pdf")

summary = {
    "method": METHOD_NAME,
    "dataset": "GSE194122 paper-main benchmark",
    "n_cells": adata.n_obs,
    "latent_shape": list(adata.obsm[LATENT_KEY].shape),
    "static_report": "benchmark_results/report.html",
    "interactive_report": "reports/interactive_report.html",
    "pdf_report": "reports/summary_report.pdf",
    "include_latent": INCLUDE_LATENT_IN_BUNDLE,
    "note": "The bundle ZIP contains the full result directory, standalone interactive HTML, PDF summary, and optionally the latent matrix plus barcodes.",
}
(ARTIFACT_DIR / "README_OUTPUT.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

archive_path = create_report_bundle(
    adata,
    result,
    WORK_DIR / "scVI_scRareBench_GSE194122_results_v0_9_1.zip",
    representation_key=LATENT_KEY,
    include_latent=INCLUDE_LATENT_IN_BUNDLE,
    write_interactive=GENERATE_INTERACTIVE_REPORT,
    write_pdf=GENERATE_PDF_REPORT,
    interactive_report_options=HTML_REPORT_OPTIONS,
)

print("Final ZIP:", archive_path)
print("ZIP size (MB):", round(archive_path.stat().st_size / 1024 / 1024, 2))


## نکات تفسیر و خروجی‌ها

- `report.html` گزارش ثابت و self-contained است.
- `interactive_report.html` dashboard کامل و مستقل benchmark است: Overview، تمام metricهای paper-style، تمام metric/statusهای scIB، Rare-cell Explorer، UMAP، Sankey، Reproducibility و Figures را در تب‌های جدا دارد.
- `summary_report.pdf` خلاصه‌ای مناسب برای اشتراک‌گذاری و آرشیو ارائه می‌کند.
- فایل ZIP نهایی کل پوشه نتایج را در کنار گزارش تعاملی، PDF و در صورت فعال بودن فلگ، فایل latent و barcodeها قرار می‌دهد.
- scVI فقط از raw counts و batch labels استفاده می‌کند؛ `celltype` فقط در benchmark مصرف می‌شود.
- بخش استاندارد همه معیارهای جاری backend `scib-metrics` را جدا از rare-cell layer نگه می‌دارد.
- امتیاز Total استاندارد scIB-compatible با rare-cell score ادغام نشده است؛ این دو خروجی علمی جدا باقی می‌مانند.
- قواعد failure archetype هنوز provisional هستند.
